# Notebook 11 — Các Đặc Trưng Của Graph (Graph Features)

Notebook này phân tích các đặc trưng cốt lõi của đồ thị, bao gồm:
1. **Đặc trưng lan truyền (Propagation)**: Sử dụng PageRank để tìm các node quan trọng.
2. **Centre Graph (Độ trung tâm - Centrality)**: Đánh giá độ trung tâm của các node (Degree Centrality).
3. **Độ tương đồng (Similarity)**: So sánh độ tương đồng giữa các node sử dụng Jaccard Similarity (dựa trên cấu trúc mạng) và Cosine Similarity (dựa trên vector đặc trưng).

In [1]:
import torch
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from torch.nn.functional import cosine_similarity
from torch_geometric.utils import to_networkx
from pathlib import Path

# Thiết lập style
sns.set_theme(style="whitegrid")

In [2]:
PROCESSED_DIR = Path("../data/processed/")
data_path = PROCESSED_DIR / "hetero_data.pt"

data = torch.load(data_path, weights_only=False)
print("Dữ liệu đồ thị:\n", data)

Dữ liệu đồ thị:
 HeteroData(
  user={ x=[1642, 384] },
  item={ x=[21639, 384] },
  (user, reviews, item)={ edge_index=[2, 10751] },
  (item, also_bought, item)={ edge_index=[2, 7253] }
)


## 1. Mạng Item-Item (Also Bought)
Để tính toán các chỉ số lan truyền và độ trung tâm, chúng ta sẽ tạo một đồ thị NetworkX thuần túy cho các Item dựa trên liên kết `also_bought`.

In [3]:
# Lấy danh sách cạnh also_bought
item_edges = data['item', 'also_bought', 'item'].edge_index.numpy()
num_items = data['item'].num_nodes

# Tạo đồ thị có hướng (Directed Graph)
G_items = nx.DiGraph()
G_items.add_nodes_from(range(num_items))
G_items.add_edges_from(zip(item_edges[0], item_edges[1]))

print(f"Đồ thị Item-Item có {G_items.number_of_nodes()} nodes và {G_items.number_of_edges()} edges.")

Đồ thị Item-Item có 21639 nodes và 7253 edges.


## 2. Đặc trưng lan truyền (Propagation) & Centre Graph
Chúng ta sẽ sử dụng **PageRank** để đánh giá khả năng lan truyền và mức độ quan trọng của các Item trong mạng lưới mua sắm, cùng với **Degree Centrality** để tìm các trung tâm của đồ thị.

In [4]:
print("Đang tính toán PageRank...")
pagerank_scores = nx.pagerank(G_items, alpha=0.85)

print("Đang tính toán Degree Centrality...")
degree_centrality = nx.degree_centrality(G_items)

# Lưu kết quả vào DataFrame để dễ quan sát
centrality_df = pd.DataFrame({
    'Item_ID': list(pagerank_scores.keys()),
    'PageRank': list(pagerank_scores.values()),
    'Degree_Centrality': list(degree_centrality.values())
})

# Sắp xếp để xem Top 5 items quan trọng nhất theo PageRank
top_pagerank = centrality_df.sort_values(by='PageRank', ascending=False).head(5)
print("\n--- Top 5 Items có PageRank cao nhất (Khả năng lan truyền tốt nhất) ---")
display(top_pagerank)

Đang tính toán PageRank...
Đang tính toán Degree Centrality...



--- Top 5 Items có PageRank cao nhất (Khả năng lan truyền tốt nhất) ---


,Item_ID,PageRank,Degree_Centrality
20357,20357,0.002586,0.002033
16178,16178,0.002414,0.001710
19356,19356,0.002108,0.007995
21031,21031,0.001839,0.006932
19496,19496,0.001829,0.006054


## 3. Độ tương đồng (Similarity)
Chúng ta sẽ chọn 2 Item bất kỳ có liên kết với nhau (ví dụ: Item `211` và một item hàng xóm của nó) để tính toán độ tương đồng.

In [5]:
item_A = 211

# Tìm một item B được mua cùng với item A
neighbors_A = list(G_items.successors(item_A))
if len(neighbors_A) > 0:
    item_B = neighbors_A[0]
else:
    item_B = 10 # Fallback
    
print(f"So sánh Item {item_A} và Item {item_B}")

So sánh Item 211 và Item 20636


### 3.1 Jaccard Similarity (Cấu trúc mạng)
Độ tương đồng Jaccard đo lường mức độ trùng lặp giữa tập hợp khách hàng (users) đã đánh giá 2 items này.

In [6]:
# Lấy danh sách users đã review Item A và Item B
reviews_edges = data['user', 'reviews', 'item'].edge_index

users_A = set(reviews_edges[0][reviews_edges[1] == item_A].numpy())
users_B = set(reviews_edges[0][reviews_edges[1] == item_B].numpy())

# Tính Jaccard Similarity: |A giao B| / |A hợp B|
intersection = len(users_A.intersection(users_B))
union = len(users_A.union(users_B))

if union == 0:
    jaccard_sim = 0.0
else:
    jaccard_sim = intersection / union

print(f"Tập User review Item {item_A}: {len(users_A)} người")
print(f"Tập User review Item {item_B}: {len(users_B)} người")
print(f"Số User review cả hai Item: {intersection} người")
print(f"\n=> Độ tương đồng Jaccard (Jaccard Similarity): {jaccard_sim:.4f}")

Tập User review Item 211: 31 người
Tập User review Item 20636: 44 người
Số User review cả hai Item: 2 người

=> Độ tương đồng Jaccard (Jaccard Similarity): 0.0274


### 3.2 Cosine Similarity (Vector đặc trưng)
Độ tương đồng Cosine đo lường góc giữa hai vector đặc trưng 384-chiều (được trích xuất từ văn bản mô tả của items).

In [7]:
# Lấy feature vectors
feature_A = data['item'].x[item_A]
feature_B = data['item'].x[item_B]

# Thêm chiều batch (1, 384) để dùng hàm cosine_similarity của PyTorch
cos_sim = cosine_similarity(feature_A.unsqueeze(0), feature_B.unsqueeze(0))

print(f"Vector đặc trưng Item {item_A}: {feature_A[:3].numpy()}... (shape: {feature_A.shape})")
print(f"Vector đặc trưng Item {item_B}: {feature_B[:3].numpy()}... (shape: {feature_B.shape})")
print(f"\n=> Độ tương đồng Cosine (Cosine Similarity): {cos_sim.item():.4f}")

Vector đặc trưng Item 211: [-0.05151524 -0.02213515  0.01965727]... (shape: torch.Size([384]))


Vector đặc trưng Item 20636: [-0.08028512  0.06363209  0.04817663]... (shape: torch.Size([384]))

=> Độ tương đồng Cosine (Cosine Similarity): 0.2646
